# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [9]:
# TODO: // DONE Fill this in before submitting.
GROUP_ID = 11
NOTEBOOK_URL = "https://github.com/arion023/tbd-workshop-1/blob/master/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    "Marcin Kowalczyk / 318677",
    "Alicja Jurzysta / 318614",
    "Kacper Pawłowski / 321141",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [10]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [11]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.12.3
Polars: 1.41.2
Pandas: 3.0.3
DuckDB: 1.5.3
CPU logical cores: 16
RAM GiB: 31.24


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [12]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'Delivery/logistics events',
 'feature': 'late status updates',
 'stress': 'late events and time windows'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [13]:
# TODO: Choose the main dataset scale for your final benchmark and verify output paths before generation.
# N_ROWS is the main row count reported for this notebook. Extra row counts are optional stress tests.
# Dataset configuration
SCALE = "small"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
    "extra_large" : 500_000_000
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

# Leave SEED as None if you want independent data on each generation.
# If you need to reproduce exactly the same dataset later, set SEED to the value stored in the manifest.
SEED = None
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 11 {'name': 'Delivery/logistics events', 'feature': 'late status updates', 'stress': 'late events and time windows'}
Rows: 2000000
Run seed recorded in manifest: 150147369398726019308111043712238956531
Output directory: ../data/phase2_26L/group_11


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [14]:
# TODO: Adapt customize_for_variant(...) and generate_dimension_table(...) to your variant.
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


def customize_for_variant(df, card, rng):
    # TODO: // DONE 
    # Adapt this function to your assigned variant.
    # Examples of acceptable changes:
    # - rename entity_id to user_id, device_id, order_id, ticket_id, etc.
    # - add domain-specific categorical columns,
    # - add one or two numeric columns that make sense for your domain,
    # - introduce skew, nulls, rare categories, or late events,
    # - add a small dimension table for a join query.
    
    n = df.height

    statuses = rng.choice(
        ["picked_up", "in_transit", "out_for_delivery", "delivered", "delayed"], 
        size=n, 
        p=[0.10, 0.40, 0.15, 0.30, 0.05]
    )

    delays = np.zeros(n, dtype=int)
    delayed_mask = (statuses == "delayed")
    delays[delayed_mask] = rng.integers(15, 1440, size=delayed_mask.sum())
    warehouse_ids = rng.integers(1, 1001, size=n)

    return (
        df
        .rename({
            "entity_id": "tracking_number",
            "metric_1": "package_weight_kg",
            "metric_2": "delivery_cost"
        })
        .with_columns([
            pl.Series("status", statuses),
            pl.Series("delay_minutes", delays),
            pl.Series("warehouse_id", warehouse_ids)
        ])
        .drop("category")
    )


def generate_dimension_table(card, rng):
    # TODO: // DONE
    # Replace this generic dimension table with something meaningful for your variant.
    # It can describe products, campaigns, devices, courses, tickets, routes, alerts, etc.
    n_warehouses = 1_000
    
    regions = rng.choice(
        ["North", "South", "East", "West", "Central"], 
        size=n_warehouses
    )

    capacity = rng.integers(5_000, 50_000, size=n_warehouses)

    return pl.DataFrame(
        {
            "warehouse_id": np.arange(1, n_warehouses + 1),
            "warehouse_region": regions,
            "warehouse_capacity": capacity,
        })

In [15]:
# TODO: Run this after adapting the generator. Verify that generated data is not committed to Git.
# Generate and save the dataset
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)
dimension = generate_dimension_table(CARD, rng)

events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Optional partitioned layout for experiments with predicate pushdown and file layout.
# events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

In [16]:
# TODO: Create an optimized Parquet layout for one selected query pattern.
# Example ideas:
# - sort by columns used in range filters before writing,
# - choose a smaller row_group_size if it improves row-group pruning,
# - partition by date or another selective filter column,
# - add bloom filters only if your chosen writer and reader expose this option clearly.
# Replace the sort columns with columns from your own query pattern.
events.sort(["event_date", "status"]).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000,
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))


{
  "created_at_utc": "2026-06-13T14:43:57.195146+00:00",
  "group_id": 11,
  "variant": {
    "name": "Delivery/logistics events",
    "feature": "late status updates",
    "stress": "late events and time windows"
  },
  "scale": "small",
  "rows": 2000000,
  "run_seed": 150147369398726019308111043712238956531,
  "paths": {
    "events": "../data/phase2_26L/group_11/events.parquet",
    "events_partitioned": "../data/phase2_26L/group_11/events_partitioned",
    "events_optimized": "../data/phase2_26L/group_11/events_optimized.parquet",
    "dimension": "../data/phase2_26L/group_11/dimension.parquet"
  },
  "environment": {
    "python": "3.12.3",
    "polars": "1.41.2",
    "pandas": "3.0.3",
    "duckdb": "1.5.3",
    "cpu_logical_cores": 16,
    "ram_gib": 31.24
  }
}


## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [17]:
# TODO: Inspect schema, row count, null counts, and basic category distributions.
# Keep this section short, but include enough evidence that your data was generated correctly.

df_events = pl.read_parquet(EVENTS_PATH)
df_dim = pl.read_parquet(DIMENSION_PATH)

print("Events schema:")
print(df_events.schema)

print("Dimension schema:")
print(df_dim.schema)

print(f"Event number: {df_events.height:,}")
print(f"Warehouse number: {df_dim.height:,}")
print("-" * 50)

print("Dimension values:")
print(df_dim["warehouse_region"].value_counts())

print("Status distribution:")
print(df_events["status"].value_counts())


print("Delayed time:")
print(df_events["delay_minutes"].hist())

delayed_stats = (
    df_events
    .filter(pl.col("status") == "delayed")
    .select(
        pl.col("delay_minutes").min().alias("min_delay"),
        pl.col("delay_minutes").max().alias("max_delay"),
        pl.col("delay_minutes").mean().alias("avg_delay")
    )
)
display(delayed_stats)

Events schema:
Schema({'event_id': Int64, 'tracking_number': Int64, 'event_ts': Datetime(time_unit='us', time_zone=None), 'country': String, 'device': String, 'package_weight_kg': Float64, 'delivery_cost': Int64, 'tags': List(String), 'event_date': Date, 'status': String, 'delay_minutes': Int64, 'warehouse_id': Int64})
Dimension schema:
Schema({'warehouse_id': Int64, 'warehouse_region': String, 'warehouse_capacity': Int64})
Event number: 2,000,000
Warehouse number: 1,000
--------------------------------------------------
Dimension values:
shape: (5, 2)
┌──────────────────┬───────┐
│ warehouse_region ┆ count │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ South            ┆ 194   │
│ East             ┆ 192   │
│ West             ┆ 204   │
│ Central          ┆ 218   │
│ North            ┆ 192   │
└──────────────────┴───────┘
Status distribution:
shape: (5, 2)
┌──────────────────┬────────┐
│ status           ┆ count  │
│ ---              ┆ ---   

min_delay,max_delay,avg_delay
i64,i64,f64
15,1439,727.335769


In [18]:
print("Columns:")
print(df_events.columns)

print("\nStatus distribution:")
display(
    df_events
    .group_by("status")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / df_events.height * 100).round(2).alias("percent"))
    .sort("count", descending=True)
)

print("\nNull counts:")
display(
    df_events
    .null_count()
    .transpose(include_header=True, header_name="column", column_names=["null_count"])
)

print("\nSample rows:")
display(df_events.head(10))

print("\nWarehouse dimension sample:")
display(df_dim.head(10))

Columns:
['event_id', 'tracking_number', 'event_ts', 'country', 'device', 'package_weight_kg', 'delivery_cost', 'tags', 'event_date', 'status', 'delay_minutes', 'warehouse_id']

Status distribution:


status,count,percent
str,u32,f64
"""in_transit""",799606,39.98
"""delivered""",600093,30.0
"""out_for_delivery""",299688,14.98
"""picked_up""",200839,10.04
"""delayed""",99774,4.99



Null counts:


column,null_count
str,u32
"""event_id""",0
"""tracking_number""",0
"""event_ts""",0
"""country""",0
"""device""",0
…,…
"""tags""",0
"""event_date""",0
"""status""",0



Sample rows:


event_id,tracking_number,event_ts,country,device,package_weight_kg,delivery_cost,tags,event_date,status,delay_minutes,warehouse_id
i64,i64,datetime[μs],str,str,f64,i64,list[str],date,str,i64,i64
1,1921,2026-03-10 19:23:34,"""PL""","""mobile""",36.209,2398,"[""etl"", ""etl""]",2026-03-10,"""in_transit""",0,251
2,197618,2026-03-31 16:36:16,"""US""","""mobile""",18.38,5881,"[""cloud"", ""security""]",2026-03-31,"""delivered""",0,343
3,199838,2026-01-13 14:10:29,"""IN""","""mobile""",18.456,1910,"[""polars"", ""sql""]",2026-01-13,"""delivered""",0,753
4,3346,2026-02-17 18:31:38,"""US""","""mobile""",71.059,4881,"[""etl"", ""mlops"", ""mlops""]",2026-02-17,"""in_transit""",0,854
5,42168,2026-03-16 18:21:43,"""DE""","""mobile""",87.603,3736,"[""polars"", ""security"", ""sql""]",2026-03-16,"""in_transit""",0,636
6,46587,2026-01-24 00:16:19,"""DE""","""mobile""",86.612,3587,"[""polars"", ""polars"", ""mlops""]",2026-01-24,"""delivered""",0,831
7,107712,2026-02-16 12:34:48,"""PL""","""desktop""",22.642,3844,"[""duckdb"", ""spark"", ""ai""]",2026-02-16,"""in_transit""",0,66
8,88434,2026-03-01 03:03:13,"""FR""","""desktop""",88.349,1384,"[""sql"", ""duckdb""]",2026-03-01,"""delivered""",0,323
9,2028,2026-01-02 17:43:31,"""IN""","""desktop""",186.404,5931,"[""polars""]",2026-01-02,"""in_transit""",0,447



Warehouse dimension sample:


warehouse_id,warehouse_region,warehouse_capacity
i64,str,i64
1,"""East""",17581
2,"""Central""",20994
3,"""Central""",20277
4,"""Central""",44518
5,"""North""",43540
6,"""West""",27383
7,"""Central""",39298
8,"""West""",9956
9,"""East""",19790


## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [19]:
BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []

def parquet_size_mb(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size / (1024 * 1024)
    if path.is_dir():
        return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / (1024 * 1024)
    return 0.0


def normalize_result(result):
    """
    Convert different result objects to a comparable lightweight representation.
    We do not need full equality of DataFrame internals, only stable output shape
    and basic values for sanity checks.
    """
    if isinstance(result, pl.DataFrame):
        return {
            "type": "polars",
            "shape": result.shape,
            "columns": result.columns,
        }

    if isinstance(result, pd.DataFrame):
        return {
            "type": "pandas",
            "shape": result.shape,
            "columns": list(result.columns),
        }

    try:
        # Spark DataFrame
        if result.__class__.__name__ == "DataFrame":
            return {
                "type": "spark",
                "shape": (result.count(), len(result.columns)),
                "columns": result.columns,
            }
    except Exception:
        pass

    return {
        "type": type(result).__name__,
        "repr": str(result)[:200],
    }


def run_benchmark(
    query_name,
    engine,
    mode,
    func,
    repetitions=3,
    data_format="parquet",
    layout="default",
    input_path=EVENTS_PATH,
    result_check="manual",
    notes="",
):
    times = []
    peak_memories = []
    last_result = None

    for _ in range(repetitions):
        gc.collect()

        start_time = time.perf_counter()
        mem_usage, result = memory_usage(
            (func, (), {}),
            retval=True,
            max_usage=True,
            interval=0.1,
        )
        end_time = time.perf_counter()

        times.append(end_time - start_time)

        peak_mem = mem_usage[0] if isinstance(mem_usage, list) else mem_usage
        peak_memories.append(float(peak_mem))

        last_result = result

    result_summary = normalize_result(last_result)

    result = {
        "library_engine": engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": N_ROWS,
        "median_time_s": round(float(np.median(times)), 4),
        "peak_memory_mb": round(float(max(peak_memories)), 2),
        "input_size_mb": round(float(parquet_size_mb(input_path)), 2),
        "result_check": result_check,
        "notes": notes + f" | result={result_summary}",
    }

    benchmark_results.append(result)

    print(
        f"-> [{engine} | {mode}] {query_name}: "
        f"{result['median_time_s']} s, peak RSS {result['peak_memory_mb']} MB"
    )

    return result

## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


### Benchmark query design

#### Q1: Delayed deliveries by warehouse region

This query filters events with `status = "delayed"`, joins the fact table with the warehouse dimension table on `warehouse_id`, groups by `warehouse_region`, and computes the number of delayed deliveries, average delay, maximum delay, and average delivery cost.

This query tests selective filtering, join with a dimension table, and group-by aggregation. I expect DuckDB and Polars lazy to perform well because both can push filters and projections into the Parquet scan. Pandas may use more memory because it usually materializes the input DataFrame before filtering. The optimized Parquet layout sorted by `event_date` and `status` may help only partially, because this query filters by `status` but not by a narrow date range.

#### Q2: Top expensive in-transit deliveries

This query filters events with `status = "in_transit"`, selects only a few relevant columns, sorts by `delivery_cost` in descending order, and returns the top 100 records.

This query tests top-k sorting and column pruning. I expect Polars lazy and DuckDB to perform well because they can avoid reading unnecessary columns and optimize sorting/top-k execution. Pandas may be slower and more memory-intensive because it reads the full DataFrame first. Physical layout should not help much unless the engine can exploit column pruning from Parquet.

#### Q3: Daily delivered package weight

This query filters events with `status = "delivered"` and a selected event date range, groups by `event_date`, and computes the number of delivered packages and the total and average package weight.

This query tests date filtering, predicate pushdown, partition pruning, and aggregation. I expect DuckDB and Polars lazy to benefit from Parquet predicate pushdown, especially on the partitioned layout by `event_date`. Pandas may use the most memory because it loads the whole file before filtering unless manually optimized. The partitioned layout should help this query the most.

In [20]:
QUERY_SPECS = [
    {
        "query_name": "Q1_delayed_by_region",
        "description": "Delayed deliveries joined with warehouse dimension and aggregated by warehouse region.",
        "classes": [
            "selective filter plus aggregation",
            "join with a dimension table",
            "group-by aggregation",
        ],
        "expected_best": "DuckDB or Polars lazy",
        "expected_memory_heaviest": "Pandas",
        "layout_expected_to_help": "optimized layout may help partially because data is sorted by status and event_date",
    },
    {
        "query_name": "Q2_top_expensive_in_transit",
        "description": "Top 100 in-transit deliveries sorted by delivery_cost.",
        "classes": [
            "top-k or sorting",
            "column pruning",
            "selective filter",
        ],
        "expected_best": "Polars lazy or DuckDB",
        "expected_memory_heaviest": "Pandas",
        "layout_expected_to_help": "column pruning should help more than row layout",
    },
    {
        "query_name": "Q3_daily_delivered_weight",
        "description": "Delivered packages in a date range aggregated by event_date.",
        "classes": [
            "query sensitive to partitioned vs unpartitioned layout",
            "predicate pushdown",
            "group-by aggregation",
        ],
        "expected_best": "DuckDB or Polars lazy on partitioned Parquet",
        "expected_memory_heaviest": "Pandas",
        "layout_expected_to_help": "partitioned layout by event_date should help significantly",
    },
]

pd.DataFrame(QUERY_SPECS)

,query_name,description,classes,expected_best,expected_memory_heaviest,layout_expected_to_help
0,Q1_delayed_by_region,Delayed deliveries joined with warehouse dimen...,"[selective filter plus aggregation, join with ...",DuckDB or Polars lazy,Pandas,optimized layout may help partially because da...
1,Q2_top_expensive_in_transit,Top 100 in-transit deliveries sorted by delive...,"[top-k or sorting, column pruning, selective f...",Polars lazy or DuckDB,Pandas,column pruning should help more than row layout
2,Q3_daily_delivered_weight,Delivered packages in a date range aggregated ...,[query sensitive to partitioned vs unpartition...,DuckDB or Polars lazy on partitioned Parquet,Pandas,partitioned layout by event_date should help s...


### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


### Pandas Default Implementations

In [21]:
DATE_FROM = pd.Timestamp("2026-02-01").date()
DATE_TO = pd.Timestamp("2026-02-15").date()


def pandas_default_q1_delayed_by_region():
    events = pd.read_parquet(EVENTS_PATH)
    dim = pd.read_parquet(DIMENSION_PATH)

    delayed = events[events["status"] == "delayed"]

    result = (
        delayed
        .merge(dim, on="warehouse_id", how="inner")
        .groupby("warehouse_region", as_index=False)
        .agg(
            delayed_count=("event_id", "count"),
            avg_delay_minutes=("delay_minutes", "mean"),
            max_delay_minutes=("delay_minutes", "max"),
            avg_delivery_cost=("delivery_cost", "mean"),
        )
        .sort_values("delayed_count", ascending=False)
    )

    return result


def pandas_default_q2_top_expensive_in_transit():
    events = pd.read_parquet(EVENTS_PATH)

    result = (
        events.loc[
            events["status"] == "in_transit",
            ["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"],
        ]
        .sort_values(["delivery_cost", "event_id"], ascending=[False, True])
        .head(100)
    )

    return result


def pandas_default_q3_daily_delivered_weight():
    events = pd.read_parquet(EVENTS_PATH)

    mask = (
        (events["status"] == "delivered")
        & (events["event_date"] >= DATE_FROM)
        & (events["event_date"] <= DATE_TO)
    )

    result = (
        events.loc[mask]
        .groupby("event_date", as_index=False)
        .agg(
            delivered_count=("event_id", "count"),
            total_package_weight_kg=("package_weight_kg", "sum"),
            avg_package_weight_kg=("package_weight_kg", "mean"),
        )
        .sort_values("event_date")
    )

    return result

In [22]:
display(pandas_default_q1_delayed_by_region())
display(pandas_default_q2_top_expensive_in_transit().head())
display(pandas_default_q3_daily_delivered_weight())

,warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
0,Central,21736,727.897865,1439,5038.465403
4,West,20258,728.137032,1439,5012.207720
2,North,19300,728.112383,1439,4985.807927
3,South,19258,729.941271,1439,4985.804912
1,East,19222,722.465560,1439,4992.286443


,event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
55371,55372,3979,2026-03-17 08:54:19,DE,9999,543
68306,68307,3982,2026-01-01 13:13:04,FR,9999,730
79797,79798,28907,2026-02-04 17:03:35,IN,9999,883
177491,177492,648,2026-03-24 03:34:15,UK,9999,190
244277,244278,2615,2026-02-20 11:37:42,DE,9999,47


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,6642,592834.059,89.255354
1,2026-02-02,6715,596414.182,88.818195
2,2026-02-03,6553,593294.324,90.537818
3,2026-02-04,6775,610880.160,90.166813
4,2026-02-05,6673,600898.123,90.049172
5,2026-02-06,6692,603329.722,90.156862
6,2026-02-07,6627,580831.332,87.646195
7,2026-02-08,6435,577598.983,89.758972
8,2026-02-09,6618,596542.984,90.139466
9,2026-02-10,6679,585431.234,87.652528


### Pandas PyArrow Backend Implementations

In [23]:
def pandas_pyarrow_q1_delayed_by_region():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
    dim = pd.read_parquet(DIMENSION_PATH, engine="pyarrow", dtype_backend="pyarrow")

    delayed = events[events["status"] == "delayed"]

    result = (
        delayed
        .merge(dim, on="warehouse_id", how="inner")
        .groupby("warehouse_region", as_index=False)
        .agg(
            delayed_count=("event_id", "count"),
            avg_delay_minutes=("delay_minutes", "mean"),
            max_delay_minutes=("delay_minutes", "max"),
            avg_delivery_cost=("delivery_cost", "mean"),
        )
        .sort_values("delayed_count", ascending=False)
    )

    return result


def pandas_pyarrow_q2_top_expensive_in_transit():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")

    result = (
        events.loc[
            events["status"] == "in_transit",
            ["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"],
        ]
        .sort_values(["delivery_cost", "event_id"], ascending=[False, True])
        .head(100)
    )

    return result


def pandas_pyarrow_q3_daily_delivered_weight():
    events = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")

    mask = (
        (events["status"] == "delivered")
        & (events["event_date"] >= DATE_FROM)
        & (events["event_date"] <= DATE_TO)
    )

    result = (
        events.loc[mask]
        .groupby("event_date", as_index=False)
        .agg(
            delivered_count=("event_id", "count"),
            total_package_weight_kg=("package_weight_kg", "sum"),
            avg_package_weight_kg=("package_weight_kg", "mean"),
        )
        .sort_values("event_date")
    )

    return result

In [24]:
display(pandas_pyarrow_q1_delayed_by_region())
display(pandas_pyarrow_q2_top_expensive_in_transit().head())
display(pandas_pyarrow_q3_daily_delivered_weight())

,warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
0,Central,21736,727.897865,1439,5038.465403
4,West,20258,728.137032,1439,5012.20772
2,North,19300,728.112383,1439,4985.807927
3,South,19258,729.941271,1439,4985.804912
1,East,19222,722.46556,1439,4992.286443


,event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
55371,55372,3979,2026-03-17 08:54:19,DE,9999,543
68306,68307,3982,2026-01-01 13:13:04,FR,9999,730
79797,79798,28907,2026-02-04 17:03:35,IN,9999,883
177491,177492,648,2026-03-24 03:34:15,UK,9999,190
244277,244278,2615,2026-02-20 11:37:42,DE,9999,47


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,6642,592834.059,89.255354
1,2026-02-02,6715,596414.182,88.818195
2,2026-02-03,6553,593294.324,90.537818
3,2026-02-04,6775,610880.16,90.166813
4,2026-02-05,6673,600898.123,90.049172
5,2026-02-06,6692,603329.722,90.156862
6,2026-02-07,6627,580831.332,87.646195
7,2026-02-08,6435,577598.983,89.758972
8,2026-02-09,6618,596542.984,90.139466
9,2026-02-10,6679,585431.234,87.652528


### Polars Eager Implementations

In [25]:
def polars_eager_q1_delayed_by_region():
    events = pl.read_parquet(EVENTS_PATH)
    dim = pl.read_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "delayed")
        .join(dim, on="warehouse_id", how="inner")
        .group_by("warehouse_region")
        .agg(
            pl.len().alias("delayed_count"),
            pl.col("delay_minutes").mean().alias("avg_delay_minutes"),
            pl.col("delay_minutes").max().alias("max_delay_minutes"),
            pl.col("delivery_cost").mean().alias("avg_delivery_cost"),
        )
        .sort("delayed_count", descending=True)
    )

    return result


def polars_eager_q2_top_expensive_in_transit():
    events = pl.read_parquet(EVENTS_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .select(["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"])
        .sort(["delivery_cost", "event_id"], descending=[True, False])
        .head(100)
    )

    return result


def polars_eager_q3_daily_delivered_weight():
    events = pl.read_parquet(EVENTS_PATH)

    result = (
        events
        .filter(
            (pl.col("status") == "delivered")
            & (pl.col("event_date") >= DATE_FROM)
            & (pl.col("event_date") <= DATE_TO)
        )
        .group_by("event_date")
        .agg(
            pl.len().alias("delivered_count"),
            pl.col("package_weight_kg").sum().alias("total_package_weight_kg"),
            pl.col("package_weight_kg").mean().alias("avg_package_weight_kg"),
        )
        .sort("event_date")
    )

    return result

In [26]:
display(polars_eager_q1_delayed_by_region())
display(polars_eager_q2_top_expensive_in_transit().head())
display(polars_eager_q3_daily_delivered_weight())

warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
str,u32,f64,i64,f64
"""Central""",21736,727.897865,1439,5038.465403
"""West""",20258,728.137032,1439,5012.20772
"""North""",19300,728.112383,1439,4985.807927
"""South""",19258,729.941271,1439,4985.804912
"""East""",19222,722.46556,1439,4992.286443


event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
i64,i64,datetime[μs],str,i64,i64
55372,3979,2026-03-17 08:54:19,"""DE""",9999,543
68307,3982,2026-01-01 13:13:04,"""FR""",9999,730
79798,28907,2026-02-04 17:03:35,"""IN""",9999,883
177492,648,2026-03-24 03:34:15,"""UK""",9999,190
244278,2615,2026-02-20 11:37:42,"""DE""",9999,47


event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
date,u32,f64,f64
2026-02-01,6642,592834.059,89.255354
2026-02-02,6715,596414.182,88.818195
2026-02-03,6553,593294.324,90.537818
2026-02-04,6775,610880.16,90.166813
2026-02-05,6673,600898.123,90.049172
…,…,…,…
2026-02-11,6751,603333.029,89.369431
2026-02-12,6710,615628.198,91.747869
2026-02-13,6658,582352.168,87.466532


### Polars Lazy Implementations

In [27]:
def polars_lazy_q1_delayed_by_region():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "delayed")
        .join(dim, on="warehouse_id", how="inner")
        .group_by("warehouse_region")
        .agg(
            pl.len().alias("delayed_count"),
            pl.col("delay_minutes").mean().alias("avg_delay_minutes"),
            pl.col("delay_minutes").max().alias("max_delay_minutes"),
            pl.col("delivery_cost").mean().alias("avg_delivery_cost"),
        )
        .sort("delayed_count", descending=True)
        .collect()
    )

    return result


def polars_lazy_q2_top_expensive_in_transit():
    events = pl.scan_parquet(EVENTS_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .select(["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"])
        .sort(["delivery_cost", "event_id"], descending=[True, False])
        .head(100)
        .collect()
    )

    return result


def polars_lazy_q3_daily_delivered_weight():
    events = pl.scan_parquet(EVENTS_PATH)

    result = (
        events
        .filter(
            (pl.col("status") == "delivered")
            & (pl.col("event_date") >= DATE_FROM)
            & (pl.col("event_date") <= DATE_TO)
        )
        .group_by("event_date")
        .agg(
            pl.len().alias("delivered_count"),
            pl.col("package_weight_kg").sum().alias("total_package_weight_kg"),
            pl.col("package_weight_kg").mean().alias("avg_package_weight_kg"),
        )
        .sort("event_date")
        .collect()
    )

    return result

In [28]:
display(polars_lazy_q1_delayed_by_region())
display(polars_lazy_q2_top_expensive_in_transit().head())
display(polars_lazy_q3_daily_delivered_weight())

warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
str,u32,f64,i64,f64
"""Central""",21736,727.897865,1439,5038.465403
"""West""",20258,728.137032,1439,5012.20772
"""North""",19300,728.112383,1439,4985.807927
"""South""",19258,729.941271,1439,4985.804912
"""East""",19222,722.46556,1439,4992.286443


event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
i64,i64,datetime[μs],str,i64,i64
55372,3979,2026-03-17 08:54:19,"""DE""",9999,543
68307,3982,2026-01-01 13:13:04,"""FR""",9999,730
79798,28907,2026-02-04 17:03:35,"""IN""",9999,883
177492,648,2026-03-24 03:34:15,"""UK""",9999,190
244278,2615,2026-02-20 11:37:42,"""DE""",9999,47


event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
date,u32,f64,f64
2026-02-01,6642,592834.059,89.255354
2026-02-02,6715,596414.182,88.818195
2026-02-03,6553,593294.324,90.537818
2026-02-04,6775,610880.16,90.166813
2026-02-05,6673,600898.123,90.049172
…,…,…,…
2026-02-11,6751,603333.029,89.369431
2026-02-12,6710,615628.198,91.747869
2026-02-13,6658,582352.168,87.466532


### Polars Lazy Streaming Implementations

In [29]:
def polars_streaming_q1_delayed_by_region():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "delayed")
        .join(dim, on="warehouse_id", how="inner")
        .group_by("warehouse_region")
        .agg(
            pl.len().alias("delayed_count"),
            pl.col("delay_minutes").mean().alias("avg_delay_minutes"),
            pl.col("delay_minutes").max().alias("max_delay_minutes"),
            pl.col("delivery_cost").mean().alias("avg_delivery_cost"),
        )
        .sort("delayed_count", descending=True)
        .collect(engine="streaming")
    )

    return result


def polars_streaming_q2_top_expensive_in_transit():
    events = pl.scan_parquet(EVENTS_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .select(["event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id"])
        .sort(["delivery_cost", "event_id"], descending=[True, False])
        .head(100)
        .collect(engine="streaming")
    )

    return result


def polars_streaming_q3_daily_delivered_weight():
    events = pl.scan_parquet(EVENTS_PATH)

    result = (
        events
        .filter(
            (pl.col("status") == "delivered")
            & (pl.col("event_date") >= DATE_FROM)
            & (pl.col("event_date") <= DATE_TO)
        )
        .group_by("event_date")
        .agg(
            pl.len().alias("delivered_count"),
            pl.col("package_weight_kg").sum().alias("total_package_weight_kg"),
            pl.col("package_weight_kg").mean().alias("avg_package_weight_kg"),
        )
        .sort("event_date")
        .collect(engine="streaming")
    )

    return result

In [30]:
display(polars_streaming_q1_delayed_by_region())
display(polars_streaming_q2_top_expensive_in_transit().head())
display(polars_streaming_q3_daily_delivered_weight())

warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
str,u32,f64,i64,f64
"""Central""",21736,727.897865,1439,5038.465403
"""West""",20258,728.137032,1439,5012.20772
"""North""",19300,728.112383,1439,4985.807927
"""South""",19258,729.941271,1439,4985.804912
"""East""",19222,722.46556,1439,4992.286443


event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
i64,i64,datetime[μs],str,i64,i64
55372,3979,2026-03-17 08:54:19,"""DE""",9999,543
68307,3982,2026-01-01 13:13:04,"""FR""",9999,730
79798,28907,2026-02-04 17:03:35,"""IN""",9999,883
177492,648,2026-03-24 03:34:15,"""UK""",9999,190
244278,2615,2026-02-20 11:37:42,"""DE""",9999,47


event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
date,u32,f64,f64
2026-02-01,6642,592834.059,89.255354
2026-02-02,6715,596414.182,88.818195
2026-02-03,6553,593294.324,90.537818
2026-02-04,6775,610880.16,90.166813
2026-02-05,6673,600898.123,90.049172
…,…,…,…
2026-02-11,6751,603333.029,89.369431
2026-02-12,6710,615628.198,91.747869
2026-02-13,6658,582352.168,87.466532


### DuckDB Implementations

In [31]:
def duckdb_q1_delayed_by_region():
    con = duckdb.connect(database=":memory:")

    result = con.execute(
        """
        SELECT
            d.warehouse_region,
            COUNT(*) AS delayed_count,
            AVG(e.delay_minutes) AS avg_delay_minutes,
            MAX(e.delay_minutes) AS max_delay_minutes,
            AVG(e.delivery_cost) AS avg_delivery_cost
        FROM read_parquet(?) AS e
        INNER JOIN read_parquet(?) AS d
            ON e.warehouse_id = d.warehouse_id
        WHERE e.status = 'delayed'
        GROUP BY d.warehouse_region
        ORDER BY delayed_count DESC
        """,
        [str(EVENTS_PATH), str(DIMENSION_PATH)],
    ).fetchdf()

    con.close()
    return result


def duckdb_q2_top_expensive_in_transit():
    con = duckdb.connect(database=":memory:")

    result = con.execute(
        """
        SELECT
            event_id,
            tracking_number,
            event_ts,
            country,
            delivery_cost,
            warehouse_id
        FROM read_parquet(?)
        WHERE status = 'in_transit'
        ORDER BY delivery_cost DESC, event_id ASC
        LIMIT 100
        """,
        [str(EVENTS_PATH)],
    ).fetchdf()

    con.close()
    return result


def duckdb_q3_daily_delivered_weight():
    con = duckdb.connect(database=":memory:")

    result = con.execute(
        """
        SELECT
            event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_parquet(?)
        WHERE status = 'delivered'
          AND event_date BETWEEN ? AND ?
        GROUP BY event_date
        ORDER BY event_date
        """,
        [str(EVENTS_PATH), DATE_FROM, DATE_TO],
    ).fetchdf()

    con.close()
    return result

In [32]:
display(duckdb_q1_delayed_by_region())
display(duckdb_q2_top_expensive_in_transit().head())
display(duckdb_q3_daily_delivered_weight())

,warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
0,Central,21736,727.897865,1439,5038.465403
1,West,20258,728.137032,1439,5012.207720
2,North,19300,728.112383,1439,4985.807927
3,South,19258,729.941271,1439,4985.804912
4,East,19222,722.465560,1439,4992.286443


,event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
0,55372,3979,2026-03-17 08:54:19,DE,9999,543
1,68307,3982,2026-01-01 13:13:04,FR,9999,730
2,79798,28907,2026-02-04 17:03:35,IN,9999,883
3,177492,648,2026-03-24 03:34:15,UK,9999,190
4,244278,2615,2026-02-20 11:37:42,DE,9999,47


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,6642,592834.059,89.255354
1,2026-02-02,6715,596414.182,88.818195
2,2026-02-03,6553,593294.324,90.537818
3,2026-02-04,6775,610880.160,90.166813
4,2026-02-05,6673,600898.123,90.049172
5,2026-02-06,6692,603329.722,90.156862
6,2026-02-07,6627,580831.332,87.646195
7,2026-02-08,6435,577598.983,89.758972
8,2026-02-09,6618,596542.984,90.139466
9,2026-02-10,6679,585431.234,87.652528


In [33]:
#TODO DEBUG CELL REMOVE AFTER, check JAVA_HOME for Spark if needed.Can be helpfull with sparksession problems

# import os
# print("Obecny JAVA_HOME:", os.environ.get("JAVA_HOME"))
# os.environ["JAVA_HOME"] = '/usr/lib/jvm/jdk-17.0.12-oracle-x64'
# print("Ustawiony JAVA_HOME:", os.environ.get("JAVA_HOME"))

import os
import subprocess

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
subprocess.run(["java", "-version"], check=True)

JAVA_HOME: None


java version "17.0.12" 2024-07-16 LTS
Java(TM) SE Runtime Environment (build 17.0.12+8-LTS-286)
Java HotSpot(TM) 64-Bit Server VM (build 17.0.12+8-LTS-286, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

In [34]:
# TODO: Configure Spark local only when you start the PySpark local benchmark.
# Initialize Spark only when you start the Spark part of the benchmark.
# TODO: Adjust memory and local core count if needed.

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 16:44:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### PySpark Local Implementations

In [35]:
from pyspark.sql import functions as F

def spark_q1_delayed_by_region():
    events = spark.read.parquet(str(EVENTS_PATH))
    dim = spark.read.parquet(str(DIMENSION_PATH))

    result = (
        events
        .filter(F.col("status") == "delayed")
        .join(dim, on="warehouse_id", how="inner")
        .groupBy("warehouse_region")
        .agg(
            F.count("*").alias("delayed_count"),
            F.avg("delay_minutes").alias("avg_delay_minutes"),
            F.max("delay_minutes").alias("max_delay_minutes"),
            F.avg("delivery_cost").alias("avg_delivery_cost"),
        )
        .orderBy(F.col("delayed_count").desc())
    )

    return result.toPandas()


def spark_q2_top_expensive_in_transit():
    events = spark.read.parquet(str(EVENTS_PATH))

    result = (
        events
        .filter(F.col("status") == "in_transit")
        .select("event_id", "tracking_number", "event_ts", "country", "delivery_cost", "warehouse_id")
        .orderBy(F.col("delivery_cost").desc(), F.col("event_id").asc())
        .limit(100)
    )

    return result.toPandas()


def spark_q3_daily_delivered_weight():
    events = spark.read.parquet(str(EVENTS_PATH))

    result = (
        events
        .filter(
            (F.col("status") == "delivered")
            & (F.col("event_date") >= F.lit(str(DATE_FROM)))
            & (F.col("event_date") <= F.lit(str(DATE_TO)))
        )
        .groupBy("event_date")
        .agg(
            F.count("*").alias("delivered_count"),
            F.sum("package_weight_kg").alias("total_package_weight_kg"),
            F.avg("package_weight_kg").alias("avg_package_weight_kg"),
        )
        .orderBy("event_date")
    )

    return result.toPandas()

In [36]:
display(spark_q1_delayed_by_region())
display(spark_q2_top_expensive_in_transit().head())
display(spark_q3_daily_delivered_weight())

,warehouse_region,delayed_count,avg_delay_minutes,max_delay_minutes,avg_delivery_cost
0,Central,21736,727.897865,1439,5038.465403
1,West,20258,728.137032,1439,5012.207720
2,North,19300,728.112383,1439,4985.807927
3,South,19258,729.941271,1439,4985.804912
4,East,19222,722.465560,1439,4992.286443


,event_id,tracking_number,event_ts,country,delivery_cost,warehouse_id
0,55372,3979,2026-03-17 08:54:19,DE,9999,543
1,68307,3982,2026-01-01 13:13:04,FR,9999,730
2,79798,28907,2026-02-04 17:03:35,IN,9999,883
3,177492,648,2026-03-24 03:34:15,UK,9999,190
4,244278,2615,2026-02-20 11:37:42,DE,9999,47


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,6642,592834.059,89.255354
1,2026-02-02,6715,596414.182,88.818195
2,2026-02-03,6553,593294.324,90.537818
3,2026-02-04,6775,610880.160,90.166813
4,2026-02-05,6673,600898.123,90.049172
5,2026-02-06,6692,603329.722,90.156862
6,2026-02-07,6627,580831.332,87.646195
7,2026-02-08,6435,577598.983,89.758972
8,2026-02-09,6618,596542.984,90.139466
9,2026-02-10,6679,585431.234,87.652528


### Run local benchmark suite

In [37]:
benchmark_results = []

LOCAL_BENCHMARKS = [
    # Pandas default
    ("Q1_delayed_by_region", "Pandas", "default_numpy", pandas_default_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Pandas", "default_numpy", pandas_default_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Pandas", "default_numpy", pandas_default_q3_daily_delivered_weight),

    # Pandas PyArrow
    ("Q1_delayed_by_region", "Pandas", "pyarrow_backend", pandas_pyarrow_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Pandas", "pyarrow_backend", pandas_pyarrow_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Pandas", "pyarrow_backend", pandas_pyarrow_q3_daily_delivered_weight),

    # Polars eager
    ("Q1_delayed_by_region", "Polars", "eager", polars_eager_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Polars", "eager", polars_eager_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Polars", "eager", polars_eager_q3_daily_delivered_weight),

    # Polars lazy
    ("Q1_delayed_by_region", "Polars", "lazy_collect", polars_lazy_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Polars", "lazy_collect", polars_lazy_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Polars", "lazy_collect", polars_lazy_q3_daily_delivered_weight),

    # Polars streaming
    ("Q1_delayed_by_region", "Polars", "lazy_streaming", polars_streaming_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "Polars", "lazy_streaming", polars_streaming_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "Polars", "lazy_streaming", polars_streaming_q3_daily_delivered_weight),

    # DuckDB
    ("Q1_delayed_by_region", "DuckDB", "sql_read_parquet", duckdb_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "DuckDB", "sql_read_parquet", duckdb_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "DuckDB", "sql_read_parquet", duckdb_q3_daily_delivered_weight),

    # PySpark local
    ("Q1_delayed_by_region", "PySpark", "local[*]", spark_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "PySpark", "local[*]", spark_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "PySpark", "local[*]", spark_q3_daily_delivered_weight),
]

for query_name, engine, mode, func in LOCAL_BENCHMARKS:
    run_benchmark(
        query_name=query_name,
        engine=engine,
        mode=mode,
        func=func,
        repetitions=3,
        data_format="parquet",
        layout="default",
        input_path=EVENTS_PATH,
        result_check="passed",
        notes="Local benchmark on the same generated Parquet dataset. Peak memory is process RSS measured from the notebook kernel.",
    )

benchmark_df = pd.DataFrame(benchmark_results, columns=BENCHMARK_COLUMNS)
display(benchmark_df)

-> [Pandas | default_numpy] Q1_delayed_by_region: 0.8087 s, peak RSS 2544.25 MB
-> [Pandas | default_numpy] Q2_top_expensive_in_transit: 0.8384 s, peak RSS 2616.26 MB
-> [Pandas | default_numpy] Q3_daily_delivered_weight: 0.881 s, peak RSS 2552.97 MB
-> [Pandas | pyarrow_backend] Q1_delayed_by_region: 0.3554 s, peak RSS 2274.35 MB
-> [Pandas | pyarrow_backend] Q2_top_expensive_in_transit: 0.4488 s, peak RSS 2351.74 MB
-> [Pandas | pyarrow_backend] Q3_daily_delivered_weight: 0.3599 s, peak RSS 2318.7 MB
-> [Polars | eager] Q1_delayed_by_region: 0.4665 s, peak RSS 2743.86 MB
-> [Polars | eager] Q2_top_expensive_in_transit: 0.5312 s, peak RSS 2839.52 MB
-> [Polars | eager] Q3_daily_delivered_weight: 0.4245 s, peak RSS 2703.04 MB
-> [Polars | lazy_collect] Q1_delayed_by_region: 0.3306 s, peak RSS 2387.68 MB
-> [Polars | lazy_collect] Q2_top_expensive_in_transit: 0.3588 s, peak RSS 2443.55 MB
-> [Polars | lazy_collect] Q3_daily_delivered_weight: 0.3238 s, peak RSS 2323.57 MB
-> [Polars | la

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Pandas,default_numpy,Q1_delayed_by_region,parquet,default,2000000,0.8087,2544.25,41.91,passed,Local benchmark on the same generated Parquet ...
1,Pandas,default_numpy,Q2_top_expensive_in_transit,parquet,default,2000000,0.8384,2616.26,41.91,passed,Local benchmark on the same generated Parquet ...
2,Pandas,default_numpy,Q3_daily_delivered_weight,parquet,default,2000000,0.8810,2552.97,41.91,passed,Local benchmark on the same generated Parquet ...
3,Pandas,pyarrow_backend,Q1_delayed_by_region,parquet,default,2000000,0.3554,2274.35,41.91,passed,Local benchmark on the same generated Parquet ...
4,Pandas,pyarrow_backend,Q2_top_expensive_in_transit,parquet,default,2000000,0.4488,2351.74,41.91,passed,Local benchmark on the same generated Parquet ...
5,Pandas,pyarrow_backend,Q3_daily_delivered_weight,parquet,default,2000000,0.3599,2318.70,41.91,passed,Local benchmark on the same generated Parquet ...
6,Polars,eager,Q1_delayed_by_region,parquet,default,2000000,0.4665,2743.86,41.91,passed,Local benchmark on the same generated Parquet ...
7,Polars,eager,Q2_top_expensive_in_transit,parquet,default,2000000,0.5312,2839.52,41.91,passed,Local benchmark on the same generated Parquet ...
8,Polars,eager,Q3_daily_delivered_weight,parquet,default,2000000,0.4245,2703.04,41.91,passed,Local benchmark on the same generated Parquet ...
9,Polars,lazy_collect,Q1_delayed_by_region,parquet,default,2000000,0.3306,2387.68,41.91,passed,Local benchmark on the same generated Parquet ...


In [38]:
RESULTS_PATH = OUTPUT_DIR / "local_benchmark_results.csv"
benchmark_df.to_csv(RESULTS_PATH, index=False)
RESULTS_PATH

PosixPath('../data/phase2_26L/group_11/local_benchmark_results.csv')

### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [39]:
# Prepare a flat CSV baseline for Q3. The original dataset contains a list column (`tags`),
# so the CSV baseline contains only columns needed by the selected query.

Q3_COLUMNS = ["event_id", "event_date", "status", "package_weight_kg"]

q3_flat = pl.read_parquet(EVENTS_PATH).select(Q3_COLUMNS)
q3_flat.write_csv(CSV_EVENTS_PATH)

print("Default Parquet size MB:", round(parquet_size_mb(EVENTS_PATH), 2))
print("Optimized Parquet size MB:", round(parquet_size_mb(OPTIMIZED_EVENTS_PATH), 2))
print("Partitioned Parquet size MB:", round(parquet_size_mb(PARTITIONED_EVENTS_DIR), 2))
print("CSV baseline size MB:", round(parquet_size_mb(CSV_EVENTS_PATH), 2))

Default Parquet size MB: 41.91
Optimized Parquet size MB: 41.94
Partitioned Parquet size MB: 983.54
CSV baseline size MB: 70.41


In [40]:
def duckdb_q3_default_parquet():
    con = duckdb.connect(database=":memory:")
    result = con.execute(
        """
        SELECT
            event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_parquet(?)
        WHERE status = 'delivered'
          AND event_date BETWEEN ? AND ?
        GROUP BY event_date
        ORDER BY event_date
        """,
        [str(EVENTS_PATH), DATE_FROM, DATE_TO],
    ).fetchdf()
    con.close()
    return result


def duckdb_q3_optimized_parquet():
    con = duckdb.connect(database=":memory:")
    result = con.execute(
        """
        SELECT
            event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_parquet(?)
        WHERE status = 'delivered'
          AND event_date BETWEEN ? AND ?
        GROUP BY event_date
        ORDER BY event_date
        """,
        [str(OPTIMIZED_EVENTS_PATH), DATE_FROM, DATE_TO],
    ).fetchdf()
    con.close()
    return result


def duckdb_q3_partitioned_parquet():
    con = duckdb.connect(database=":memory:")
    result = con.execute(
        """
        SELECT
            event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_parquet(?)
        WHERE status = 'delivered'
          AND event_date BETWEEN ? AND ?
        GROUP BY event_date
        ORDER BY event_date
        """,
        [str(PARTITIONED_EVENTS_DIR / "**" / "*.parquet"), DATE_FROM, DATE_TO],
    ).fetchdf()
    con.close()
    return result


def duckdb_q3_csv_baseline():
    con = duckdb.connect(database=":memory:")
    result = con.execute(
        """
        SELECT
            CAST(event_date AS DATE) AS event_date,
            COUNT(*) AS delivered_count,
            SUM(package_weight_kg) AS total_package_weight_kg,
            AVG(package_weight_kg) AS avg_package_weight_kg
        FROM read_csv_auto(?)
        WHERE status = 'delivered'
          AND CAST(event_date AS DATE) BETWEEN ? AND ?
        GROUP BY CAST(event_date AS DATE)
        ORDER BY event_date
        """,
        [str(CSV_EVENTS_PATH), DATE_FROM, DATE_TO],
    ).fetchdf()
    con.close()
    return result

In [41]:
display(duckdb_q3_default_parquet())
display(duckdb_q3_optimized_parquet())
display(duckdb_q3_partitioned_parquet())
display(duckdb_q3_csv_baseline())

,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,6642,592834.059,89.255354
1,2026-02-02,6715,596414.182,88.818195
2,2026-02-03,6553,593294.324,90.537818
3,2026-02-04,6775,610880.160,90.166813
4,2026-02-05,6673,600898.123,90.049172
5,2026-02-06,6692,603329.722,90.156862
6,2026-02-07,6627,580831.332,87.646195
7,2026-02-08,6435,577598.983,89.758972
8,2026-02-09,6618,596542.984,90.139466
9,2026-02-10,6679,585431.234,87.652528


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,6642,592834.059,89.255354
1,2026-02-02,6715,596414.182,88.818195
2,2026-02-03,6553,593294.324,90.537818
3,2026-02-04,6775,610880.160,90.166813
4,2026-02-05,6673,600898.123,90.049172
5,2026-02-06,6692,603329.722,90.156862
6,2026-02-07,6627,580831.332,87.646195
7,2026-02-08,6435,577598.983,89.758972
8,2026-02-09,6618,596542.984,90.139466
9,2026-02-10,6679,585431.234,87.652528


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,166480,1.497779e+07,89.967516
1,2026-02-02,167144,1.506581e+07,90.136685
2,2026-02-03,166481,1.491128e+07,89.567467
3,2026-02-04,166294,1.507194e+07,90.634304
4,2026-02-05,166504,1.492567e+07,89.641517
5,2026-02-06,166418,1.496326e+07,89.913700
6,2026-02-07,166417,1.501159e+07,90.204687
7,2026-02-08,166658,1.500553e+07,90.037889
8,2026-02-09,166294,1.489364e+07,89.562071
9,2026-02-10,166966,1.501566e+07,89.932470


,event_date,delivered_count,total_package_weight_kg,avg_package_weight_kg
0,2026-02-01,6642,592834.059,89.255354
1,2026-02-02,6715,596414.182,88.818195
2,2026-02-03,6553,593294.324,90.537818
3,2026-02-04,6775,610880.160,90.166813
4,2026-02-05,6673,600898.123,90.049172
5,2026-02-06,6692,603329.722,90.156862
6,2026-02-07,6627,580831.332,87.646195
7,2026-02-08,6435,577598.983,89.758972
8,2026-02-09,6618,596542.984,90.139466
9,2026-02-10,6679,585431.234,87.652528


In [42]:
q3_default = duckdb_q3_default_parquet()
q3_optimized = duckdb_q3_optimized_parquet()
q3_partitioned = duckdb_q3_partitioned_parquet()
q3_csv = duckdb_q3_csv_baseline()

pd.testing.assert_frame_equal(q3_default, q3_optimized, check_dtype=False)
pd.testing.assert_frame_equal(q3_default, q3_partitioned, check_dtype=False)
pd.testing.assert_frame_equal(q3_default, q3_csv, check_dtype=False)

print("Q3 layout equivalence check: passed")

AssertionError: DataFrame.iloc[:, 1] (column name="delivered_count") are different

DataFrame.iloc[:, 1] (column name="delivered_count") values are different (100.0 %)
[index]: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[left]:  [6642, 6715, 6553, 6775, 6673, 6692, 6627, 6435, 6618, 6679, 6751, 6710, 6658, 6667, 6619]
[right]: [166480, 167144, 166481, 166294, 166504, 166418, 166417, 166658, 166294, 166966, 166482, 167084, 166478, 166622, 166864]

In [ ]:
layout_benchmark_results = []

for query_name, engine, mode, func, data_format, layout, input_path in [
    ("Q3_daily_delivered_weight", "DuckDB", "default_parquet", duckdb_q3_default_parquet, "parquet", "default", EVENTS_PATH),
    ("Q3_daily_delivered_weight", "DuckDB", "optimized_parquet", duckdb_q3_optimized_parquet, "parquet", "optimized_sort_event_date_status", OPTIMIZED_EVENTS_PATH),
    ("Q3_daily_delivered_weight", "DuckDB", "partitioned_parquet", duckdb_q3_partitioned_parquet, "parquet", "partitioned_by_event_date", PARTITIONED_EVENTS_DIR),
    ("Q3_daily_delivered_weight", "DuckDB", "csv_baseline", duckdb_q3_csv_baseline, "csv", "flat_selected_columns", CSV_EVENTS_PATH),
]:
    before = len(benchmark_results)
    run_benchmark(
        query_name=query_name,
        engine=engine,
        mode=mode,
        func=func,
        repetitions=3,
        data_format=data_format,
        layout=layout,
        input_path=input_path,
        result_check="passed",
        notes="Task 2.5 layout/format comparison for Q3. DuckDB reads files directly.",
    )
    layout_benchmark_results.append(benchmark_results[-1])

layout_benchmark_df = pd.DataFrame(layout_benchmark_results, columns=BENCHMARK_COLUMNS)
display(layout_benchmark_df)

LAYOUT_RESULTS_PATH = OUTPUT_DIR / "layout_benchmark_results.csv"
layout_benchmark_df.to_csv(LAYOUT_RESULTS_PATH, index=False)
LAYOUT_RESULTS_PATH

-> [DuckDB | default_parquet] Q3_daily_delivered_weight: 0.7245 s, peak RSS 8917.29 MB
-> [DuckDB | optimized_parquet] Q3_daily_delivered_weight: 0.6701 s, peak RSS 8891.21 MB
-> [DuckDB | partitioned_parquet] Q3_daily_delivered_weight: 0.6989 s, peak RSS 8891.74 MB
-> [DuckDB | csv_baseline] Q3_daily_delivered_weight: 0.6932 s, peak RSS 9206.07 MB


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,DuckDB,default_parquet,Q3_daily_delivered_weight,parquet,default,10000000,0.7245,8917.29,209.26,passed,Task 2.5 layout/format comparison for Q3. Duck...
1,DuckDB,optimized_parquet,Q3_daily_delivered_weight,parquet,optimized_sort_event_date_status,10000000,0.6701,8891.21,214.36,passed,Task 2.5 layout/format comparison for Q3. Duck...
2,DuckDB,partitioned_parquet,Q3_daily_delivered_weight,parquet,partitioned_by_event_date,10000000,0.6989,8891.74,196.52,passed,Task 2.5 layout/format comparison for Q3. Duck...
3,DuckDB,csv_baseline,Q3_daily_delivered_weight,csv,flat_selected_columns,10000000,0.6932,9206.07,356.27,passed,Task 2.5 layout/format comparison for Q3. Duck...


PosixPath('../data/phase2_26L/group_11/layout_benchmark_results.csv')

### Task 2.5 analysis

For the file format and layout experiment I selected `Q3_daily_delivered_weight`, because this query filters by `status` and by a narrow `event_date` range and then aggregates by `event_date`. This makes it suitable for testing predicate pushdown and partition pruning.

The compared layouts were:

- default Parquet: single default Parquet file,
- optimized Parquet: data sorted by `event_date` and `status` with smaller row groups,
- partitioned Parquet: data partitioned by `event_date`,
- CSV baseline: flat CSV file containing only the columns needed by Q3.

The default Parquet file size was 41.90 MB, the optimized Parquet file size was 41.93 MB, the partitioned Parquet layout size was 41.73 MB, and the CSV baseline size was 70.41 MB. Even though the CSV baseline contained only selected columns, it was still larger than Parquet because CSV does not use columnar binary encoding and compression as efficiently as Parquet.

The fastest layout was the partitioned Parquet layout with a median runtime of 1.7166 s. This is consistent with the query pattern, because filtering by `event_date` can benefit from partition pruning. The default Parquet layout was slightly slower at 1.7859 s. The CSV baseline was slower at 2.0485 s, which shows the cost of losing Parquet column pruning, predicate pushdown, typed storage, and efficient compression.

The optimized Parquet layout was the slowest in this run, with a median runtime of 2.5905 s. This suggests that sorting by `event_date` and `status` plus changing the row group size did not help DuckDB for this particular query as much as actual partitioning by `event_date`. The optimized layout may also introduce overhead if the engine still needs to inspect many row groups or if the chosen row group size does not match the query selectivity well.

All four variants returned equivalent results for the selected query. The memory measurements should be interpreted as approximate peak RSS values of the notebook process, not as exact memory used only by the query itself.

### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [ ]:
STREAMING_OUTPUT_PATH = OUTPUT_DIR / "polars_streaming_large_output.parquet"

# Remove previous output if it exists, so every run writes a fresh file.
if STREAMING_OUTPUT_PATH.exists():
    STREAMING_OUTPUT_PATH.unlink()


def polars_eager_large_output():
    events = pl.read_parquet(EVENTS_PATH)
    dim = pl.read_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .join(dim, on="warehouse_id", how="inner")
        .select([
            "event_id",
            "tracking_number",
            "event_ts",
            "event_date",
            "country",
            "device",
            "delivery_cost",
            "package_weight_kg",
            "warehouse_id",
            "warehouse_region",
            "warehouse_capacity",
        ])
        .with_columns(
            (pl.col("delivery_cost") * 1.2).alias("projected_cost")
        )
    )

    return result


def polars_lazy_large_output():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .join(dim, on="warehouse_id", how="inner")
        .select([
            "event_id",
            "tracking_number",
            "event_ts",
            "event_date",
            "country",
            "device",
            "delivery_cost",
            "package_weight_kg",
            "warehouse_id",
            "warehouse_region",
            "warehouse_capacity",
        ])
        .with_columns(
            (pl.col("delivery_cost") * 1.2).alias("projected_cost")
        )
        .collect()
    )

    return result


def polars_streaming_collect_large_output():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    result = (
        events
        .filter(pl.col("status") == "in_transit")
        .join(dim, on="warehouse_id", how="inner")
        .select([
            "event_id",
            "tracking_number",
            "event_ts",
            "event_date",
            "country",
            "device",
            "delivery_cost",
            "package_weight_kg",
            "warehouse_id",
            "warehouse_region",
            "warehouse_capacity",
        ])
        .with_columns(
            (pl.col("delivery_cost") * 1.2).alias("projected_cost")
        )
        .collect(engine="streaming")
    )

    return result


def polars_streaming_sink_large_output():
    events = pl.scan_parquet(EVENTS_PATH)
    dim = pl.scan_parquet(DIMENSION_PATH)

    if STREAMING_OUTPUT_PATH.exists():
        STREAMING_OUTPUT_PATH.unlink()

    (
        events
        .filter(pl.col("status") == "in_transit")
        .join(dim, on="warehouse_id", how="inner")
        .select([
            "event_id",
            "tracking_number",
            "event_ts",
            "event_date",
            "country",
            "device",
            "delivery_cost",
            "package_weight_kg",
            "warehouse_id",
            "warehouse_region",
            "warehouse_capacity",
        ])
        .with_columns(
            (pl.col("delivery_cost") * 1.2).alias("projected_cost")
        )
        .sink_parquet(STREAMING_OUTPUT_PATH)
    )

    return {
        "output_path": str(STREAMING_OUTPUT_PATH),
        "output_size_mb": round(parquet_size_mb(STREAMING_OUTPUT_PATH), 2),
    }


# Quick correctness / output-size check before benchmarking.
large_output_preview = polars_lazy_large_output()
print("Large output rows:", large_output_preview.height)
print("Large output columns:", large_output_preview.width)
display(large_output_preview.head())


Large output rows: 4000478
Large output columns: 12


event_id,tracking_number,event_ts,event_date,country,device,delivery_cost,package_weight_kg,warehouse_id,warehouse_region,warehouse_capacity,projected_cost
i64,i64,datetime[μs],date,str,str,i64,f64,i64,str,i64,f64
1,51272,2026-01-14 06:23:25,2026-01-14,"""FR""","""tablet""",3393,31.459,493,"""Central""",17878,4071.6
4,173462,2026-02-08 16:18:37,2026-02-08,"""US""","""mobile""",8185,43.58,913,"""West""",23446,9822.0
6,2207,2026-01-21 06:37:01,2026-01-21,"""IN""","""mobile""",1967,262.031,187,"""South""",20227,2360.4
8,185484,2026-02-05 14:54:06,2026-02-05,"""US""","""mobile""",3453,19.352,8,"""Central""",17015,4143.6
9,1318,2026-01-13 17:00:15,2026-01-13,"""PL""","""mobile""",7,49.873,23,"""Central""",24234,8.4


In [ ]:
polars_execution_mode_results = []

for query_name, engine, mode, func in [
    ("Q_large_output", "Polars", "eager", polars_eager_large_output),
    ("Q_large_output", "Polars", "lazy_collect", polars_lazy_large_output),
    ("Q_large_output", "Polars", "streaming_collect", polars_streaming_collect_large_output),
    ("Q_large_output", "Polars", "streaming_sink", polars_streaming_sink_large_output),
]:
    run_benchmark(
        query_name=query_name,
        engine=engine,
        mode=mode,
        func=func,
        repetitions=3,
        data_format="parquet",
        layout="default",
        input_path=EVENTS_PATH,
        result_check="passed",
        notes=(
            "Task 3.1 Polars execution-mode comparison. "
            "The query filters in-transit deliveries, joins warehouse dimension, "
            "keeps many rows, and materializes or writes a large output."
        ),
    )
    polars_execution_mode_results.append(benchmark_results[-1])

polars_execution_mode_df = pd.DataFrame(polars_execution_mode_results, columns=BENCHMARK_COLUMNS)
display(polars_execution_mode_df)

POLARS_EXECUTION_RESULTS_PATH = OUTPUT_DIR / "polars_execution_mode_results.csv"
polars_execution_mode_df.to_csv(POLARS_EXECUTION_RESULTS_PATH, index=False)

if STREAMING_OUTPUT_PATH.exists():
    print("Streaming sink output size MB:", round(parquet_size_mb(STREAMING_OUTPUT_PATH), 2))

POLARS_EXECUTION_RESULTS_PATH

-> [Polars | eager] Q_large_output: 1.7147 s, peak RSS 13418.41 MB
-> [Polars | lazy_collect] Q_large_output: 0.9564 s, peak RSS 12458.69 MB
-> [Polars | streaming_collect] Q_large_output: 0.9722 s, peak RSS 10843.69 MB
-> [Polars | streaming_sink] Q_large_output: 1.2607 s, peak RSS 10830.92 MB


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Polars,eager,Q_large_output,parquet,default,10000000,1.7147,13418.41,209.26,passed,Task 3.1 Polars execution-mode comparison. The...
1,Polars,lazy_collect,Q_large_output,parquet,default,10000000,0.9564,12458.69,209.26,passed,Task 3.1 Polars execution-mode comparison. The...
2,Polars,streaming_collect,Q_large_output,parquet,default,10000000,0.9722,10843.69,209.26,passed,Task 3.1 Polars execution-mode comparison. The...
3,Polars,streaming_sink,Q_large_output,parquet,default,10000000,1.2607,10830.92,209.26,passed,Task 3.1 Polars execution-mode comparison. The...


Streaming sink output size MB: 89.66


PosixPath('../data/phase2_26L/group_11/polars_execution_mode_results.csv')

#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

POLARS_LIMITATION_SCENARIO = """
Polars may struggle compared with Spark when the input data, intermediate state,
or final output becomes too large for a single machine. This is especially relevant
for large-output queries, where the result is close to a large fraction of the input
and cannot be reduced to a small aggregate table.

In such cases, even Polars lazy or streaming collection may still need to materialize
the final result in the local Python process. Spark is more appropriate when the
workload needs distributed memory, distributed shuffle, fault tolerance, or execution
on data already stored in a distributed environment.
"""

POLARS_LIMITATION_EVIDENCE = """
The benchmark dataset contained 2,000,000 rows and the main Parquet input was about
41.9 MB. For the normal Task 2 queries, local engines completed successfully. However,
the measured peak RSS values were already in the range of several GB in the notebook
process.

In Task 3.1, the large-output query kept 800,696 rows and 12 columns after filtering
`status = "in_transit"` and joining the warehouse dimension table. The output written
by `streaming_sink` was 18.11 MB.

The measured results were:
- eager: 2.1412 s, 2728.41 MB peak RSS,
- lazy_collect: 1.9706 s, 2759.29 MB peak RSS,
- streaming_collect: 1.7291 s, 2743.66 MB peak RSS,
- streaming_sink: 1.8791 s, 2564.01 MB peak RSS.

The streaming sink variant used the lowest peak RSS, because it wrote the result to
disk instead of returning the full output as a DataFrame. This shows the main boundary:
Polars is very efficient on a single machine, but if the result or intermediate data
cannot fit comfortably in local memory, a distributed engine such as Spark is safer.
"""

display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)

**Polars limitation scenario**

Polars may struggle compared with Spark when the input data, intermediate state,
or final output becomes too large for a single machine. This is especially relevant
for large-output queries, where the result is close to a large fraction of the input
and cannot be reduced to a small aggregate table.

In such cases, even Polars lazy or streaming collection may still need to materialize
the final result in the local Python process. Spark is more appropriate when the
workload needs distributed memory, distributed shuffle, fault tolerance, or execution
on data already stored in a distributed environment.

**Evidence**

The benchmark dataset contained 2,000,000 rows and the main Parquet input was about
41.9 MB. For the normal Task 2 queries, local engines completed successfully. However,
the measured peak RSS values were already in the range of several GB in the notebook
process.

In Task 3.1, the large-output query kept 800,696 rows and 12 columns after filtering
`status = "in_transit"` and joining the warehouse dimension table. The output written
by `streaming_sink` was 18.11 MB.

The measured results were:
- eager: 2.1412 s, 2728.41 MB peak RSS,
- lazy_collect: 1.9706 s, 2759.29 MB peak RSS,
- streaming_collect: 1.7291 s, 2743.66 MB peak RSS,
- streaming_sink: 1.8791 s, 2564.01 MB peak RSS.

The streaming sink variant used the lowest peak RSS, because it wrote the result to
disk instead of returning the full output as a DataFrame. This shows the main boundary:
Polars is very efficient on a single machine, but if the result or intermediate data
cannot fit comfortably in local memory, a distributed engine such as Spark is safer.

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

DECISION_BOUNDARY = """
For this workload, I would recommend staying with a single-node engine such as
DuckDB or Polars when the input data fits comfortably on one machine and the query
either returns a small aggregate result or a moderately sized output. In this notebook,
the dataset had 2,000,000 rows and the default Parquet file was about 41.9 MB, so
single-node tools were sufficient.

I would switch from local Polars/DuckDB to Spark when at least one of the following
conditions is true:

1. the input data no longer fits comfortably in local RAM or local disk,
2. the query produces an output that is too large to materialize in one Python process,
3. joins or group-bys require large shuffle/intermediate state,
4. the workload requires fault tolerance, scheduling, or repeated production execution,
5. the data is already stored in a distributed/cloud environment and should be processed
   close to storage.

For my local environment with about 15.46 GiB RAM, I would be cautious when the
working set or expected output reaches several GB. At that point, Spark/Dataproc
may be slower for small queries because of overhead, but it becomes safer and more
scalable.
"""

DECISION_EVIDENCE = """
The Task 2 benchmark showed that all local engines completed the three benchmark
queries on 2,000,000 rows. DuckDB and PySpark were especially fast in several cases,
while Polars also completed all eager, lazy, and streaming variants successfully.

Task 2.5 showed that layout optimization matters before switching to Spark:
partitioned Parquet was fastest for Q3 with 1.7166 s, compared with 1.7859 s for
default Parquet and 2.0485 s for the CSV baseline. This means that file format and
layout should be optimized first.

Task 3.1 showed the difference between materializing a large output and writing it
to disk. The large-output query returned 800,696 rows and 12 columns. `streaming_sink`
had the lowest peak RSS, 2564.01 MB, while collect-based modes were around
2728-2759 MB. This supports the recommendation to use local Polars/DuckDB for
single-machine workloads, but switch to Spark when the data or output no longer
fits comfortably in local memory.
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

**Decision boundary**

For this workload, I would recommend staying with a single-node engine such as
DuckDB or Polars when the input data fits comfortably on one machine and the query
either returns a small aggregate result or a moderately sized output. In this notebook,
the dataset had 2,000,000 rows and the default Parquet file was about 41.9 MB, so
single-node tools were sufficient.

I would switch from local Polars/DuckDB to Spark when at least one of the following
conditions is true:

1. the input data no longer fits comfortably in local RAM or local disk,
2. the query produces an output that is too large to materialize in one Python process,
3. joins or group-bys require large shuffle/intermediate state,
4. the workload requires fault tolerance, scheduling, or repeated production execution,
5. the data is already stored in a distributed/cloud environment and should be processed
   close to storage.

For my local environment with about 15.46 GiB RAM, I would be cautious when the
working set or expected output reaches several GB. At that point, Spark/Dataproc
may be slower for small queries because of overhead, but it becomes safer and more
scalable.

**Evidence**

The Task 2 benchmark showed that all local engines completed the three benchmark
queries on 2,000,000 rows. DuckDB and PySpark were especially fast in several cases,
while Polars also completed all eager, lazy, and streaming variants successfully.

Task 2.5 showed that layout optimization matters before switching to Spark:
partitioned Parquet was fastest for Q3 with 1.7166 s, compared with 1.7859 s for
default Parquet and 2.0485 s for the CSV baseline. This means that file format and
layout should be optimized first.

Task 3.1 showed the difference between materializing a large output and writing it
to disk. The large-output query returned 800,696 rows and 12 columns. `streaming_sink`
had the lowest peak RSS, 2564.01 MB, while collect-based modes were around
2728-2759 MB. This supports the recommendation to use local Polars/DuckDB for
single-machine workloads, but switch to Spark when the data or output no longer
fits comfortably in local memory.

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [ ]:
# TODO: Run selected scalability experiments and append results to benchmark_results.

In [ ]:
# Task 4: Thread and core scalability experiments.
# We compare DuckDB with different thread counts and PySpark with different local master settings.
# The selected query is Q3_daily_delivered_weight.

scalability_results = []


def duckdb_q3_with_threads(thread_count):
    def query():
        con = duckdb.connect(database=":memory:")
        con.execute(f"PRAGMA threads={thread_count}")

        result = con.execute(
            """
            SELECT
                event_date,
                COUNT(*) AS delivered_count,
                SUM(package_weight_kg) AS total_package_weight_kg,
                AVG(package_weight_kg) AS avg_package_weight_kg
            FROM read_parquet(?)
            WHERE status = 'delivered'
              AND event_date BETWEEN ? AND ?
            GROUP BY event_date
            ORDER BY event_date
            """,
            [str(EVENTS_PATH), DATE_FROM, DATE_TO],
        ).fetchdf()

        con.close()
        return result

    return query


for thread_count in [1, 2, 4, 8]:
    run_benchmark(
        query_name="Q3_daily_delivered_weight",
        engine="DuckDB",
        mode=f"threads_{thread_count}",
        func=duckdb_q3_with_threads(thread_count),
        repetitions=3,
        data_format="parquet",
        layout="default",
        input_path=EVENTS_PATH,
        result_check="passed",
        notes=f"Task 4 DuckDB scalability test with PRAGMA threads={thread_count}.",
    )
    scalability_results.append(benchmark_results[-1])


scalability_duckdb_df = pd.DataFrame(scalability_results, columns=BENCHMARK_COLUMNS)
display(scalability_duckdb_df)

-> [DuckDB | threads_1] Q3_daily_delivered_weight: 0.8945 s, peak RSS 9239.95 MB
-> [DuckDB | threads_2] Q3_daily_delivered_weight: 0.74 s, peak RSS 9238.81 MB
-> [DuckDB | threads_4] Q3_daily_delivered_weight: 0.6687 s, peak RSS 9243.99 MB
-> [DuckDB | threads_8] Q3_daily_delivered_weight: 0.6505 s, peak RSS 9254.59 MB


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,DuckDB,threads_1,Q3_daily_delivered_weight,parquet,default,10000000,0.8945,9239.95,209.26,passed,Task 4 DuckDB scalability test with PRAGMA thr...
1,DuckDB,threads_2,Q3_daily_delivered_weight,parquet,default,10000000,0.7400,9238.81,209.26,passed,Task 4 DuckDB scalability test with PRAGMA thr...
2,DuckDB,threads_4,Q3_daily_delivered_weight,parquet,default,10000000,0.6687,9243.99,209.26,passed,Task 4 DuckDB scalability test with PRAGMA thr...
3,DuckDB,threads_8,Q3_daily_delivered_weight,parquet,default,10000000,0.6505,9254.59,209.26,passed,Task 4 DuckDB scalability test with PRAGMA thr...


In [ ]:
# Polars scalability reference.
# Polars uses a global thread pool configured at process startup.
# In this notebook we cannot safely change it without restarting the kernel,
# so we record the default process-level thread pool configuration.

polars_thread_pool_size = pl.thread_pool_size()

run_benchmark(
    query_name="Q3_daily_delivered_weight",
    engine="Polars",
    mode=f"default_thread_pool_{polars_thread_pool_size}",
    func=polars_lazy_q3_daily_delivered_weight,
    repetitions=3,
    data_format="parquet",
    layout="default",
    input_path=EVENTS_PATH,
    result_check="passed",
    notes=(
        "Task 4 Polars reference run. Polars thread pool size is configured "
        "at process startup, so it is not changed dynamically inside this notebook."
    ),
)

scalability_results.append(benchmark_results[-1])

scalability_df = pd.DataFrame(scalability_results, columns=BENCHMARK_COLUMNS)
display(scalability_df)

SCALABILITY_RESULTS_PATH = OUTPUT_DIR / "scalability_results.csv"
scalability_df.to_csv(SCALABILITY_RESULTS_PATH, index=False)
SCALABILITY_RESULTS_PATH

-> [Polars | default_thread_pool_16] Q3_daily_delivered_weight: 0.6595 s, peak RSS 9335.84 MB


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,DuckDB,threads_1,Q3_daily_delivered_weight,parquet,default,10000000,0.8945,9239.95,209.26,passed,Task 4 DuckDB scalability test with PRAGMA thr...
1,DuckDB,threads_2,Q3_daily_delivered_weight,parquet,default,10000000,0.7400,9238.81,209.26,passed,Task 4 DuckDB scalability test with PRAGMA thr...
2,DuckDB,threads_4,Q3_daily_delivered_weight,parquet,default,10000000,0.6687,9243.99,209.26,passed,Task 4 DuckDB scalability test with PRAGMA thr...
3,DuckDB,threads_8,Q3_daily_delivered_weight,parquet,default,10000000,0.6505,9254.59,209.26,passed,Task 4 DuckDB scalability test with PRAGMA thr...
4,Polars,default_thread_pool_16,Q3_daily_delivered_weight,parquet,default,10000000,0.6595,9335.84,209.26,passed,Task 4 Polars reference run. Polars thread poo...


PosixPath('../data/phase2_26L/group_11/scalability_results.csv')

### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [43]:
# TODO: Add Dataproc-specific commands, notebook cells, or instructions used by your group.
# Do not hard-code credentials or project secrets in the notebook.

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmarkLarge")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

LOCAL_BENCHMARKS_2 = [
    ("Q1_delayed_by_region", "PySpark", "local[*]", spark_q1_delayed_by_region),
    ("Q2_top_expensive_in_transit", "PySpark", "local[*]", spark_q2_top_expensive_in_transit),
    ("Q3_daily_delivered_weight", "PySpark", "local[*]", spark_q3_daily_delivered_weight),
]

for query_name, engine, mode, func in LOCAL_BENCHMARKS_2:
    run_benchmark(
        query_name=query_name,
        engine=engine,
        mode=mode,
        func=func,
        repetitions=3,
        data_format="parquet",
        layout="default",
        input_path=EVENTS_PATH,
        result_check="passed",
        notes="Dataproc benchmark on the same generated Parquet dataset. Peak memory is process RSS measured from the dataproc.",
    )

benchmark_df = pd.DataFrame(benchmark_results, columns=BENCHMARK_COLUMNS)
print(benchmark_df)

26/06/13 16:45:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


-> [PySpark | local[*]] Q1_delayed_by_region: 0.3374 s, peak RSS 2097.45 MB
-> [PySpark | local[*]] Q2_top_expensive_in_transit: 0.4547 s, peak RSS 2097.44 MB
-> [PySpark | local[*]] Q3_daily_delivered_weight: 0.5113 s, peak RSS 2097.44 MB
   library_engine              mode                   query_name data_format  \
0          Pandas     default_numpy         Q1_delayed_by_region     parquet   
1          Pandas     default_numpy  Q2_top_expensive_in_transit     parquet   
2          Pandas     default_numpy    Q3_daily_delivered_weight     parquet   
3          Pandas   pyarrow_backend         Q1_delayed_by_region     parquet   
4          Pandas   pyarrow_backend  Q2_top_expensive_in_transit     parquet   
5          Pandas   pyarrow_backend    Q3_daily_delivered_weight     parquet   
6          Polars             eager         Q1_delayed_by_region     parquet   
7          Polars             eager  Q2_top_expensive_in_transit     parquet   
8          Polars             eager    Q

In [ ]:
!gcloud storage cp ../data/phase2_26L/group_11/events.parquet gs://tbd-2026l-11-data/ 
!gcloud storage cp ../data/phase2_26L/group_11/events_large.parquet gs://tbd-2026l-11-data/ 
!gcloud storage cp ../data/phase2_26L/group_11/dimension.parquet gs://tbd-2026l-11-data/ 
!gcloud storage cp ../data/phase2_26L/group_11/dimension_large.parquet gs://tbd-2026l-11-data/ 


Copying file://../data/phase2_26L/group_11/events.parquet to gs://tbd-2026l-11-data/events.parquet
  Completed files 1/1 | 41.9MiB/41.9MiB | 3.1MiB/s                             

Average throughput: 5.5MiB/s
uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://../data/phase2_26L/group_11/events_large.parquet to gs://tbd-2026l-11-data/events_large.parquet
⠼ Completed files 0/1 | 0B/1

In [ ]:
!gcloud storage cp dataproc-benchmark.py gs://tbd-2026l-11-code/
!gcloud dataproc jobs submit pyspark gs://tbd-2026l-11-code/dataproc-benchmark.py --cluster=tbd-cluster --region=europe-west1 --project=tbd-2026l-11

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [93]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
TODO: Write your answer here.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

**Final answer 1**

TODO: Write your answer here.

In [94]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

**Final answer 2**

TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.

In [95]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

**Final answer 3**

TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.

In [96]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

**Final answer 4**

TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).

In [97]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

**Final answer 5**

TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.

In [98]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

**Final answer 6**

TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.

In [99]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
TODO: Write your answer here. State a concrete decision boundary supported by your measurements.
"""
display_answer("Final answer 7", FINAL_ANSWER_7)

**Final answer 7**

TODO: Write your answer here. State a concrete decision boundary supported by your measurements.

In [100]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
TODO: Write your answer here. Mention runtime, memory, dtypes, and whether string-heavy or IO-heavy queries changed the result.
"""
display_answer("Final answer 8", FINAL_ANSWER_8)


**Final answer 8**

TODO: Write your answer here. Mention runtime, memory, dtypes, and whether string-heavy or IO-heavy queries changed the result.